In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Fri Oct  2 21:15:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb" \
    "evaluate"

In [4]:
import os
import math
import random
import json
import logging
from datetime import datetime

import torch
import torch.nn as nn
import wandb
import evaluate
from transformers import (
    AutoModelForQuestionAnswering,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, 
    DataCollatorWithPadding,
    AutoTokenizer,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    TrainerState,
)
from peft import LoraConfig
from safetensors.torch import load_file, save_file
from datasets import load_dataset, Dataset

# Configuration

In [5]:
# Run configuration
TASK = 'squad'  # 'wikipedia' | 'squad'
LANG = 'en'     # wikipedia: 'en' | '<any_lang>', squad: 'en' only
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data configuration
MLM_PROB = 0.15
# --- 3.5K (80%:20%) ---
# TRAIN_SIZE = 3_500
# VAL_SIZE = 875
# --- 15K (80%:20%) ---
TRAIN_SIZE = 15_000
VAL_SIZE = 3_750

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
MODEL_NAME = 'XLM-R-Base'
# --- New training ---
# RESUME_MODEL_ID = None
# RESUME_CKPT_STEP = None
# --- Resume training ---
RESUME_MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047'
RESUME_CKPT_STEP = 6000

# LegameX configuration
# --- Reference LoRA (trained on T4) ---
REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260929201303'
REF_LORA_CKPT_STEP = 1000
# --- Reference LoRA (trained on RTX 3090) ---
# REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260922211643'
# REF_LORA_CKPT_STEP = 600
# --- Gate ---
GATE_RANK = 1
GATE_ALPHA = 1
GATE_WARMUP_STEPS = 0

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LR = 1e-4
# --- wikipedia, 2~4K ---
# WARMUP_STEPS = 50
# EVAL_STEPS = 100
# SAVE_STEPS = 100
# LOGGING_STEPS = 50
# ---
# NUM_EPOCHS = 3
# EARLY_STOPPING_PATIENCE = 3
# ---
# NUM_EPOCHS = 10
# EARLY_STOPPING_PATIENCE = 5
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 5
# --- squad, 15K ---
WARMUP_STEPS = 100
EVAL_STEPS = 300
SAVE_STEPS = 300
LOGGING_STEPS = 100
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 3
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 5
# ---
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5
# ---
# NUM_EPOCHS = 15
# EARLY_STOPPING_PATIENCE = 10

In [6]:
# Resume training configuration
resume_from_checkpoint = None
if RESUME_MODEL_ID:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
else:
    run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-s{SEED}-LegameX-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    
    # Automatically retrieve the username of the currently logged-in user
    from huggingface_hub import HfApi
    user_info = HfApi().whoami()
    username = user_info["name"]

    hub_model_id = f"{username}/{run_name}"
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-mrg-v{version}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 193 files:   0%|          | 0/193 [00:00<?, ?it/s]

checkpoint-6300/legamex.pt:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

checkpoint-6300/rng_state.pth:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

checkpoint-6300/optimizer.pt:   0%|          | 0.00/4.29M [00:00<?, ?B/s]

...

checkpoint-6300/training_args.bin:   0%|          | 0.00/5.91k [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/5.91k [00:00<?, ?B/s]

runs/Oct02_20-44-42_7ec507cc57b1/events.(…):   0%|          | 0.00/6.20k [00:00<?, ?B/s]

Resume from checkpoint: alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047/checkpoint-6000
Model name: XLM-R-Base
Run name: XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047
Hub model ID: alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047
Hub merged model ID: alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-mrg-v261001202047


# Utilities

In [7]:
# LoRA utilities
def download_hf_model(
        repo_id, 
        ckpt_step, 
        max_checkpoints=10_000,
        ckpt_interval=25,
    ):
    local_dir = repo_id.split('/')[-1]
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]
        ignore_checkpoints += ['runs/*', '**/wandb/*']

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = os.path.join(local_dir, f'checkpoint-{ckpt_step}')
    return local_dir, ckpt_dir

def check_parameter(n, p):
    print(f"{'name':<8}:", n)
    print(f"{'device':<8}:", p.device)
    print(f"{'dtype':<8}:", p.dtype)
    print(f"{'norm':<8}:", p.norm().item())
    print(f"{'mean':<8}:", p.mean().item())
    # print(f"{'min':<8}:", p.min().item())
    # print(f"{'max':<8}:", p.max().item())

def check_weights(model, infix, limit=1):
    for i, (n, p) in enumerate(model.named_parameters()):
        if infix in n:
            check_parameter(n, p)
            print()
            if i >= limit - 1:
                break

# Data utilities
def load_train_val_datasets(
    lang, # e.g., 'en' | 'ja' | 'id'
    task, # 'wikipedia' | 'squad'
    train_size, val_size,
    token=None,
):
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
            token=token,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Model

In [8]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the appropriate model classes based on the task
if TASK == 'squad':
    model_cls = AutoModelForQuestionAnswering
    task_type = 'QUESTION_ANS'
    data_collator = DataCollatorWithPadding(tokenizer)
    label_names = ['start_positions', 'end_positions']
else:
    model_cls = AutoModelForMaskedLM
    task_type = 'FEATURE_EXTRACTION'
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROB,
    )
    label_names = ['labels']

# Load the base model
base_model = model_cls.from_pretrained(MODEL_ID)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
class LegamexLora(nn.Module):
    def __init__(
            self, base_module, ref_lora_config, tfr_lora_config,
                    
            # Gate parameters
            gate_warmup_steps, gate_scheduler_type='cosine', 
            gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
            gate_bias=False, gate_use_rslora=False,
        ):
        super().__init__()

        assert gate_scheduler_type in ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt'], \
            f"Unknown gate scheduler type: {gate_scheduler_type}"
        
        self.base_module = base_module
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.training_step = 0
        
        # Determine in_features and out_features from base module
        in_features = getattr(base_module, 'in_features', None)
        out_features = getattr(base_module, 'out_features', None)
        if in_features is None or out_features is None:
            raise ValueError(f"Cannot determine in_features or out_features from {base_module}.")
        
        # Initialize reference (ref) and transfer (tfr) LoRA modules
        self.lora = nn.ModuleDict({
            'ref': self._init_lora_module(in_features, out_features, 
                                          rank=ref_lora_config.r, 
                                          alpha=ref_lora_config.lora_alpha, 
                                          dropout=ref_lora_config.lora_dropout, 
                                          bias=False if ref_lora_config.bias == 'none' else True, 
                                          use_rslora=ref_lora_config.use_rslora),
            'tfr': self._init_lora_module(in_features, out_features, 
                                          rank=tfr_lora_config.r, 
                                          alpha=tfr_lora_config.lora_alpha, 
                                          dropout=tfr_lora_config.lora_dropout, 
                                          bias=False if tfr_lora_config.bias == 'none' else True, 
                                          use_rslora=tfr_lora_config.use_rslora),
        })
        
        # Initialize gate modules
        self.gate = self._init_lora_module(in_features, out_features, 
                                           rank=gate_rank, alpha=gate_alpha, 
                                           dropout=gate_dropout, bias=gate_bias, 
                                           use_rslora=gate_use_rslora)
        self.gate_act = nn.Sigmoid()
    
    def _init_lora_module(self, in_features, out_features, rank, alpha, dropout, bias, use_rslora):
            device = self.base_module.weight.device
            dropout_module = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
            scaling = alpha / math.sqrt(rank) if use_rslora else alpha / rank
            
            # Initialize LoRA modules with extracted parameters
            lora_A = nn.Linear(in_features, rank, bias=bias, device=device)
            lora_B = nn.Linear(rank, out_features, bias=bias, device=device)
            
            # Initialize weights for LoRA modules
            # nn.init.normal_(lora_A.weight, mean=0.0, std=1 / math.sqrt(rank))
            nn.init.kaiming_uniform_(lora_A.weight, a=math.sqrt(5))
            nn.init.zeros_(lora_B.weight)
            
            lora_module = nn.ModuleDict({
                'A': lora_A,
                'B': lora_B,
                'dropout': dropout_module,
            })
            lora_module.scaling = scaling
            lora_module.use_bias = bias
            return lora_module

    def load_lora_weights(self, state_dict, prefix, lora_type):
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        device_A = self.lora[lora_type].A.weight.device # type: ignore
        device_B = self.lora[lora_type].B.weight.device # type: ignore

        key_A = f'{prefix}.lora_A.weight'
        key_B = f'{prefix}.lora_B.weight'

        with torch.no_grad():
            self.lora[lora_type].A.weight.copy_(state_dict[key_A].to(device_A)) # type: ignore
            self.lora[lora_type].B.weight.copy_(state_dict[key_B].to(device_B)) # type: ignore

            if self.lora[lora_type].use_bias:
                if f'{prefix}.lora_A.bias' in state_dict:
                    self.lora[lora_type].A.bias.copy_(state_dict[f'{prefix}.lora_A.bias'].to(device_A)) # type: ignore
                if f'{prefix}.lora_B.bias' in state_dict:
                    self.lora[lora_type].B.bias.copy_(state_dict[f'{prefix}.lora_B.bias'].to(device_B)) # type: ignore    
    
    def gate_scheduler(self):
        start = 0.0
        end = 1.0
        step = self.training_step
        warmup_steps = self.gate_warmup_steps
        scheduler_type = self.gate_scheduler_type
        
        if warmup_steps <= 0:
            return end

        t = min(max(step / warmup_steps, 0.0), 1.0)

        if scheduler_type == 'linear':
            p = t
        elif scheduler_type == 'cosine':
            # cosine warmup
            p = 0.5 * (1.0 - math.cos(math.pi * t))
        elif scheduler_type == 'quadratic':
            p = t**2
        elif scheduler_type == 'cubic':
            p = t**3
        elif scheduler_type == 'sqrt':
            p = math.sqrt(t)
        else:
            raise ValueError(f"Unknown gate scheduler type: {scheduler_type}")

        return start + (end - start) * p
        
    def forward(self, x):
        # Compute base layer output
        base_out = self.base_module(x)
        
        # Check if input tensor needs to be converted to LoRA module's dtype
        requires_conversion = not torch.is_autocast_enabled()
        if requires_conversion:
            # TODO: Add assertion to check if all weights have same dtype
            x = x.to(self.lora.tfr.A.weight.dtype) # pyright: ignore
        
        # Compute outputs for reference and transfer LoRA
        ref_lora_out = self.lora.ref.B(self.lora.ref.dropout(self.lora.ref.A(x))) * self.lora.ref.scaling # type: ignore
        tfr_lora_out = self.lora.tfr.B(self.lora.tfr.dropout(self.lora.tfr.A(x))) * self.lora.tfr.scaling # type: ignore
        
        # Compute gate output and its complement
        gate_out = self.gate_act(self.gate.B(self.gate.dropout(self.gate.A(x)))) # type: ignore
        gate_out = gate_out * self.gate_scheduler() # Schedule gate output based on training step and warmup steps
        gate_comp_out = torch.ones(gate_out.shape, dtype=gate_out.dtype, device=gate_out.device) - gate_out
        
        # Compute LoRA output
        lora_out = gate_out * ref_lora_out + gate_comp_out * tfr_lora_out
        
        # Check if LoRA output needs to be converted back to base layer's dtype
        if requires_conversion:
            lora_out = lora_out.to(base_out.dtype)
            
        return base_out + lora_out
        
    def set_training_step(self, training_step):
        self.training_step = training_step

class LegamexLoraModel(nn.Module):
    def __init__(
            self, base_model, task_type, ref_lora_config, tfr_lora_config, 
            
            # Gate parameters
            gate_warmup_steps, gate_scheduler_type='cosine', 
            gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
            gate_bias=False, gate_use_rslora=False
        ):
        super().__init__()
        
        # Determine base model name or path
        self.base_model_name_or_path = (
            ref_lora_config.base_model_name_or_path
            if hasattr(ref_lora_config, 'base_model_name_or_path')
            else base_model.config.name_or_path
        )
        if base_model.config.name_or_path != self.base_model_name_or_path:
            print("Using base model name from reference LoRA:", self.base_model_name_or_path)
        
        # Store arguments
        self.base_model = base_model
        self.task_type = task_type
        self.ref_lora_config = ref_lora_config
        self.tfr_lora_config = tfr_lora_config
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.gate_rank = gate_rank
        self.gate_alpha = gate_alpha
        self.gate_dropout = gate_dropout
        self.gate_bias = gate_bias
        self.gate_use_rslora = gate_use_rslora

        # Wrap target modules with LegameX
        self.legamex_modules = {}
        self._wrap_target_modules_with_legamex()

    def _wrap_target_modules_with_legamex(self):
        assert self.ref_lora_config.target_modules == self.tfr_lora_config.target_modules, \
            "Reference and transfer LoRA must have same target modules."
        target_modules = self.tfr_lora_config.target_modules
        
        for module_name, module in self.base_model.named_modules():
            # Skip modules already wrapped with LegamexLora
            if isinstance(module, LegamexLora):
                # module_name = module_name.rsplit('model.', 1)[-1]
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                continue
            
            if any(target_module in module_name for target_module in target_modules):
                # Wrap module with LegamexLora
                parent_module, child_name = self._get_parent_module(module_name)
                legamex_module = LegamexLora(
                    base_module=module,
                    ref_lora_config=self.ref_lora_config,
                    tfr_lora_config=self.tfr_lora_config,
                    gate_warmup_steps=self.gate_warmup_steps,
                    gate_scheduler_type=self.gate_scheduler_type,
                    gate_rank=self.gate_rank,
                    gate_alpha=self.gate_alpha,
                    gate_dropout=self.gate_dropout,
                    gate_bias=self.gate_bias,
                    gate_use_rslora=self.gate_use_rslora,
                )
                setattr(parent_module, child_name, legamex_module)
                
                # Store LegamexLora module for later weight loading
                # module_name = module_name.rsplit('model.', 1)[-1]
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = legamex_module
                
            # Store qa_outputs modules if task_type is 'QUESTION_ANS'
            if isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':
                # module_name = module_name.rsplit('model.', 1)[-1]
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                
        # Freeze all modules,n unfreeze all LegameX-specific modules except reference LoRA
        self.freeze_all(verbose=False)
        self.unfreeze_legamex_except_ref_lora(verbose=True)

    def _get_parent_module(self, module_name):
        parts = module_name.split('.')
        parent_module = self.base_model
        for part in parts[:-1]:
            parent_module = getattr(parent_module, part)
        return parent_module, parts[-1]

    def _json_safe(self, obj):
        if isinstance(obj, set):
            return list(obj)
        if isinstance(obj, dict):
            return {k: self._json_safe(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [self._json_safe(v) for v in obj]
        return obj
    
    def forward(self, *args, **kwargs):
        kwargs.pop('num_items_in_batch', None) # Remove Trainer-only kwargs before forwarding to base model
        return self.base_model.forward(*args, **kwargs)

    def freeze_all(self, verbose=False):
        for p in self.base_model.parameters():
            p.requires_grad = False
        if verbose:
            print("All modules are frozen.")

    def unfreeze_legamex_except_ref_lora(self, verbose=False):
        for module_name, module in self.legamex_modules.items():
            if isinstance(module, LegamexLora):
                # Unfreeze lora.tfr weights and biases
                module.lora.tfr.A.weight.requires_grad = True # type: ignore
                module.lora.tfr.B.weight.requires_grad = True # type: ignore
                if module.lora.tfr.use_bias: # type: ignore
                    if hasattr(module.lora.tfr.A, 'bias') and module.lora.tfr.A.bias is not None: # type: ignore
                        module.lora.tfr.A.bias.requires_grad = True # type: ignore
                    if hasattr(module.lora.tfr.B, 'bias') and module.lora.tfr.B.bias is not None: # type: ignore
                        module.lora.tfr.B.bias.requires_grad = True # type: ignore
                
                # Unfreeze gate weights and biases
                module.gate.A.weight.requires_grad = True # type: ignore
                module.gate.B.weight.requires_grad = True # type: ignore
                if module.gate.use_bias:
                    if hasattr(module.gate.A, 'bias') and module.gate.A.bias is not None: # type: ignore
                        module.gate.A.bias.requires_grad = True # type: ignore
                    if hasattr(module.gate.B, 'bias') and module.gate.B.bias is not None: # type: ignore
                        module.gate.B.bias.requires_grad = True # type: ignore

            # Unfreeze qa_outputs weight and bias if task_type is 'QUESTION_ANS'
            elif isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':    
                module.weight.requires_grad = True
                if hasattr(module, 'bias') and module.bias is not None:
                        module.bias.requires_grad = True

        if verbose:
            trainable = sum(p.numel() for p in self.base_model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.base_model.parameters())
            print(f"Trainable parameters: {trainable:,} / {total:,}")

    def load_lora_weights(self, lora_path, lora_type):
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        if not os.path.exists(lora_path):
            raise FileNotFoundError(f"LoRA weights file not found: {lora_path}")
        
        if lora_path.endswith('.safetensors'):
            state_dict = load_file(lora_path)
        else:
            state_dict = torch.load(lora_path, map_location='cpu')
            
        processed_module_names = set()
        for key in state_dict.keys():
            key_prefix, key_rest = key.rsplit('model.', 1)
            module_name = key_rest.split('.lora', 1)[0]
            module_name_sanitized = module_name.replace('.', '__DOT__')
            
            if module_name_sanitized in processed_module_names:
                continue
            
            if module_name_sanitized in self.legamex_modules:
                prefix = key_prefix + 'model.' + module_name
                self.legamex_modules[module_name_sanitized].load_lora_weights(state_dict, prefix, lora_type)
            else:
                print("LoRA weights not found for module:", module_name)
                
            processed_module_names.add(module_name_sanitized)

        print(f"LoRA weights loaded from {lora_path} into model.")

    @classmethod
    def from_pretrained(cls, save_dir, **kwargs):
        # Load LegameX configuration
        with open(os.path.join(save_dir, 'legamex_config.json'), 'r') as f:
            config = json.load(f)
            
        # Recreate base model
        base_model_id = config['base_model_name_or_path']
        task_type = config['task_type']
        if task_type == 'QUESTION_ANS':
            base_model = AutoModelForQuestionAnswering.from_pretrained(base_model_id)
        elif task_type == 'FEATURE_EXTRACTION':
            base_model = AutoModelForMaskedLM.from_pretrained(base_model_id)
        else:
            raise ValueError(f"Unknown task type: {task_type}")
        
        ref_lora_config = LoraConfig(**config['ref_lora_config'])
        tfr_lora_config = LoraConfig(**config['tfr_lora_config'])
        
        # Instantiate LegamexLoraModel (wraps base model)
        model = cls(
            base_model=base_model,
            task_type=task_type,
            ref_lora_config=ref_lora_config,
            tfr_lora_config=tfr_lora_config,
            gate_warmup_steps=config['gate_warmup_steps'],
            gate_scheduler_type=config['gate_scheduler_type'],
            gate_rank=config['gate_rank'],
            gate_alpha=config['gate_alpha'],
            gate_dropout=config['gate_dropout'],
            gate_bias=config['gate_bias'],
            gate_use_rslora=config['gate_use_rslora'],
        )
        
        full_state_dict = torch.load(os.path.join(save_dir, 'legamex.pt'), map_location='cpu')

        for key, tensor in full_state_dict.items():
            parts = key.split('.', 1) # 'module_name' + 'rest.of.path'
            if len(parts) == 2:
                module_name, param_name = parts
                module = model.legamex_modules.get(module_name)
                if module is not None:
                    # Look up parameter inside module using a flat dict
                    param = dict(module.named_parameters()).get(param_name)
                    if param is not None:
                        param.data.copy_(tensor)
        
        return model
        
    def save_pretrained(self, save_dir):
        # Create save directories
        os.makedirs(save_dir, exist_ok=True)
        tfr_dir = os.path.join(save_dir, 'tfr')
        os.makedirs(tfr_dir, exist_ok=True)
        
        # Store LegameX and transfer state dicts
        state_dict = {}
        tfr_state_dict = {}
        
        for module_name, module in self.legamex_modules.items():
            for param_name, param in module.named_parameters():
                # Skip base_module parameters
                if 'base_module' in param_name:
                    continue
                
                # Store the rest of parameters in LegameX state dict
                full_param_name = f'{module_name}.{param_name}'
                param_cpu = param.detach().cpu()
                state_dict[full_param_name] = param_cpu
                
                # Store transfer component and qa_outputs parameters separately
                if 'lora.tfr' in param_name or 'qa_outputs' in module_name:
                    full_param_name_sanitized = full_param_name.replace('__DOT__', '.').replace('lora.tfr.', 'lora_')
                    tfr_param_cpu = param_cpu

                    # If parameter is lora.tfr weight, merge it with gate complement weight
                    if 'lora.tfr' in param_name and 'weight' in param_name:
                        gate_weight_name = param_name.replace('lora.tfr', 'gate')
                        
                        try:
                            gate_param = module.get_parameter(gate_weight_name)
                            gate_weight = gate_param.detach().cpu()
                            gate_comp_weight = 1.0 - gate_weight
                            tfr_param_cpu = tfr_param_cpu * gate_comp_weight
                        except AttributeError:
                            raise AttributeError(
                                f"Could not find matching gate weight component '{gate_weight_name}' "
                                f"in module '{module_name}' to pair with transfer weights."
                            )
                        
                    tfr_state_dict[full_param_name_sanitized] = tfr_param_cpu
                    
        torch.save(state_dict, os.path.join(save_dir, 'legamex.pt'))
        
        # Save transfer state dict and its configuration
        save_file(tfr_state_dict, os.path.join(tfr_dir, 'adapter_model.safetensors'))
        with open(os.path.join(tfr_dir, 'adapter_config.json'), 'w') as f:
            json.dump(self._json_safe(self.tfr_lora_config.to_dict()), f, indent=4)
        
        # Save LegameX configuration for rebuilding wrapper
        config = {
            'base_model_name_or_path': self.base_model_name_or_path,
            'task_type': self.task_type,
            'ref_lora_config': self._json_safe(self.ref_lora_config.to_dict()),
            'tfr_lora_config': self._json_safe(self.tfr_lora_config.to_dict()),
            'gate_warmup_steps': self.gate_warmup_steps,
            'gate_scheduler_type': self.gate_scheduler_type,
            'gate_rank': self.gate_rank,
            'gate_alpha': self.gate_alpha,
            'gate_dropout': self.gate_dropout,
            'gate_bias': self.gate_bias,
            'gate_use_rslora': self.gate_use_rslora,
        }
        with open(os.path.join(save_dir, 'legamex_config.json'), 'w') as f:
            json.dump(config, f, indent=4)

    def set_training_step(self, training_step):
        for module in self.legamex_modules.values():
            if hasattr(module, 'set_training_step'):
                module.set_training_step(training_step)
    
    def __getattr__(self, name):
        try:
            return super().__getattr__(name) # Try getting attribute from self
        except AttributeError:
            return getattr(self.base_model, name) # Fallback to base_model
    
# Load reference LoRA configuration
ref_lora_dir = REF_LORA_ID
if len(REF_LORA_ID.split('/')) == 2:
    _, downloaded_ref_lora_dir = download_hf_model(REF_LORA_ID, REF_LORA_CKPT_STEP)
    if not downloaded_ref_lora_dir:
        raise RuntimeError(f"Could not download reference LoRA: {REF_LORA_ID}")
    ref_lora_dir = downloaded_ref_lora_dir
ref_lora_path = os.path.join(ref_lora_dir, 'adapter_model.safetensors')
ref_lora_config = LoraConfig.from_pretrained(ref_lora_dir)

# Initialize LegameX model
legamex_model = LegamexLoraModel(
    base_model,
    task_type=task_type,
    ref_lora_config=ref_lora_config,
    tfr_lora_config=ref_lora_config, # Set to be the same as reference LoRA
    gate_warmup_steps=GATE_WARMUP_STEPS,
)
legamex_model.to(DEVICE)

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Trainable parameters: 1,494,530 / 280,274,690


LegamexLoraModel(
  (base_model): XLMRobertaForQuestionAnswering(
    (roberta): XLMRobertaModel(
      (embeddings): XLMRobertaEmbeddings(
        (word_embeddings): Embedding(250002, 768, padding_idx=1)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): XLMRobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x XLMRobertaLayer(
            (attention): XLMRobertaAttention(
              (self): XLMRobertaSdpaSelfAttention(
                (query): LegamexLora(
                  (base_module): Linear(in_features=768, out_features=768, bias=True)
                  (lora): ModuleDict(
                    (ref): ModuleDict(
                      (A): Linear(in_features=768, out_features=8, bias=False)
                      (B): Linear(in_features=8, out_features=76

In [10]:
print("Check unloaded LegameX-specific weights:")
check_weights(legamex_model, infix='lora.ref')
check_weights(legamex_model, infix='lora.tfr')
check_weights(legamex_model, infix='gate')

legamex_model.load_lora_weights(ref_lora_path, lora_type='ref')
print()

print("Check loaded LegameX-specific weights:")
check_weights(legamex_model, infix='lora.ref')
check_weights(legamex_model, infix='lora.tfr')
check_weights(legamex_model, infix='gate')

Check unloaded LegameX-specific weights:
name    : base_model.roberta.encoder.layer.0.attention.self.query.lora.ref.A.weight
device  : cuda:0
dtype   : torch.float32
norm    : 1.6253881454467773
mean    : 1.9226874428568408e-05

name    : base_model.roberta.encoder.layer.0.attention.self.query.lora.tfr.A.weight
device  : cuda:0
dtype   : torch.float32
norm    : 1.6159838438034058
mean    : 4.710128268925473e-05

name    : base_model.roberta.encoder.layer.0.attention.self.query.gate.A.weight
device  : cuda:0
dtype   : torch.float32
norm    : 0.5649282336235046
mean    : 0.00043216440826654434

LoRA weights loaded from XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260929201303/checkpoint-1000/adapter_model.safetensors into model.

Check loaded LegameX-specific weights:
name    : base_model.roberta.encoder.layer.0.attention.self.query.lora.ref.A.weight
device  : cuda:0
dtype   : torch.float32
norm    : 1.7079386711120605
mean    : -0.000366502528777346

name    : base_model.roberta.encoder.layer.0

In [11]:
# Sanity check: ensure that the only trainable parameters are the transfer LoRA, gate, and qa_outputs weights and biases
# for n, p in legamex_model.named_parameters():
#     print(n, p.requires_grad)

# Data

In [12]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(
    lang=LANG, task=TASK,
    train_size=TRAIN_SIZE, val_size=VAL_SIZE
)

print("Training dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)

Training dataset:
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 15000
})

Validation dataset:
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 3750
})


In [13]:
# Preprocess the dataset
WIKI_MAX_LENGTH = 510  # 510 + 2 (BOS + EOS) = 512, matching XLM-R's max_position_embeddings
NUM_CHUNKS_PER_WIKI_ARTICLE = 3

# Bookkeeping data for string-level SQuAD evaluation
#
# IMPORTANT: datasets.map(batched=True) invokes preprocess_squad once per
# BATCH (default batch_size=1000), so the bookkeeping must be ACCUMULATED
# across batches. Overwriting it here would leave only the LAST batch's
# features, which is shorter than the full eval predictions and crashes
# compute_metrics_squad with IndexError: list index out of range.
_squad_eval_bookkeeping = {
    'feature_example_ids': [],
    'feature_offset_mappings': [],
    'feature_contexts': [],
    'references': [],
}

if TASK == 'squad':

    def preprocess_squad(examples, save_bookkeeping=False):
        tokenized = tokenizer(
            examples['question'],
            examples['context'],
            truncation='only_second', # Only truncate the context
            max_length=384,
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding=False,
        )

        # Each generated window points back to its original SQuAD example.
        sample_mapping = tokenized.pop('overflow_to_sample_mapping') # Global index

        # Save offsets separately because the model does not need them.
        offset_mapping = tokenized.pop('offset_mapping') # Local index

        start_positions = []
        end_positions = []

        # Indices of windows that contain the complete answer.
        keep_indices = []

        for feature_idx, offsets in enumerate(offset_mapping):

            # Which original example produced this window?
            sample_idx = sample_mapping[feature_idx]

            answer = examples['answers'][sample_idx]

            answer_start_char = answer['answer_start'][0]
            answer_text = answer['text'][0]
            answer_end_char = (
                answer_start_char + len(answer_text)
            )

            # sequence_ids tells us which tokens belong to:
            # 0 = question
            # 1 = context
            # None = special tokens
            sequence_ids = tokenized.sequence_ids(feature_idx)

            # Find the first and last context tokens in this window.
            context_token_indices = [
                idx
                for idx, seq_id in enumerate(sequence_ids)
                if seq_id == 1
            ]

            if not context_token_indices:
                continue

            context_start = context_token_indices[0]
            context_end = context_token_indices[-1]

            # Check whether the COMPLETE answer is inside this window.
            #
            # If the answer starts before the first context token,
            # or ends after the last context token, this window
            # does not contain the complete answer.
            if offsets[context_start][0] > answer_start_char:
                continue

            if offsets[context_end][1] < answer_end_char:
                continue

            # Find the token containing the answer start.
            token_start = context_start

            while (
                token_start <= context_end
                and offsets[token_start][1] <= answer_start_char
            ):
                token_start += 1

            # Find the token containing the answer end.
            token_end = context_end

            while (
                token_end >= context_start
                and offsets[token_end][0] >= answer_end_char
            ):
                token_end -= 1

            # Safety check.
            if token_start > context_end or token_end < context_start:
                continue

            # Keep this window.
            keep_indices.append(feature_idx)
            start_positions.append(token_start)
            end_positions.append(token_end)

        # Keep only windows containing the complete answer.
        tokenized = {
            key: [
                values[i]
                for i in keep_indices
            ]
            for key, values in tokenized.items()
        }

        tokenized['start_positions'] = start_positions
        tokenized['end_positions'] = end_positions

        # Save bookkeeping data for string-level SQuAD evaluation
        if save_bookkeeping:
            # Extend (do NOT overwrite): .map() calls this function once per
            # batch, and the mapped dataset concatenates each batch's features
            # in call order, so appending keeps bookkeeping aligned with the
            # final dataset rows.
            _squad_eval_bookkeeping['feature_example_ids'].extend(
                examples['id'][sample_mapping[i]] for i in keep_indices
            )
            _squad_eval_bookkeeping['feature_offset_mappings'].extend(
                offset_mapping[i] for i in keep_indices
            )
            _squad_eval_bookkeeping['feature_contexts'].extend(
                examples['context'][sample_mapping[i]] for i in keep_indices
            )

            # Build references (one per unique example), deduped across batches
            seen = {
                ref['id']
                for ref in _squad_eval_bookkeeping['references']
            }
            for i in keep_indices:
                ex_id = examples['id'][sample_mapping[i]]
                if ex_id not in seen:
                    seen.add(ex_id)
                    _squad_eval_bookkeeping['references'].append({
                        'id': ex_id,
                        'answers': examples['answers'][sample_mapping[i]],
                    })

        return tokenized


    train_dataset = train_dataset.map(
        preprocess_squad,
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        lambda examples: preprocess_squad(examples, save_bookkeeping=True),
        batched=True,
        remove_columns=val_dataset.column_names,
        # The bookkeeping is a SIDE EFFECT of this map call: never serve it
        # from the datasets cache, otherwise the function is skipped and the
        # bookkeeping stays empty/stale.
        load_from_cache_file=False,
    )

else:

    def tokenize_text(examples, indices):
        rng = random.Random(SEED)

        all_input_ids = []
        all_attention_masks = []
        all_labels = []

        for text, article_idx in zip(examples['text'], indices):

            # Tokenize the entire article without truncation.
            # Special tokens are added after chunking.
            article_tokens = tokenizer(
                text,
                truncation=False,
                add_special_tokens=False,
            )['input_ids']

            # Create non-overlapping 512-token chunks.
            chunks = [
                article_tokens[i:i + WIKI_MAX_LENGTH]
                for i in range(
                    0,
                    len(article_tokens),
                    WIKI_MAX_LENGTH,
                )
            ]

            # Keep only complete 512-token chunks.
            chunks = [
                chunk
                for chunk in chunks
                if len(chunk) == WIKI_MAX_LENGTH
            ]

            # Select up to N chunks randomly.
            num_chunks = min(
                NUM_CHUNKS_PER_WIKI_ARTICLE,
                len(chunks),
            )

            if num_chunks == 0:
                continue

            # Use a deterministic per-article RNG.
            article_rng = random.Random(
                SEED + article_idx
            )

            selected_chunks = article_rng.sample(
                chunks,
                num_chunks,
            )

            for chunk in selected_chunks:

                # Add special tokens in the same way
                # the XLM-R tokenizer normally does.
                input_ids = (
                    [tokenizer.bos_token_id]
                    + chunk
                    + [tokenizer.eos_token_id]
                )
                attention_mask = [1] * len(input_ids)

                all_input_ids.append(input_ids)
                all_attention_masks.append(attention_mask)
                all_labels.append(input_ids.copy())

        return {
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'labels': all_labels,
        }

    train_dataset = train_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=val_dataset.column_names,
    )

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3750 [00:00<?, ? examples/s]

In [14]:
print("Total preprocessed training samples:", len(train_dataset))
print("Total preprocessed validation samples:", len(val_dataset))

Total preprocessed training samples: 15048
Total preprocessed validation samples: 3757


# Training

In [15]:
logger = logging.getLogger(__name__)

class LegamexTrainer(Trainer):
    def _save(self, output_dir=None, state_dict=None):
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        # Use our own lightweight saver instead of Trainer's default full state_dict dump
        self.model.save_pretrained(output_dir)

        # Still save processing_class/tokenizer + training args like Trainer normally would
        if self.processing_class is not None:
            self.processing_class.save_pretrained(output_dir)
        torch.save(self.args, os.path.join(output_dir, 'training_args.bin'))
        
    def _load_from_checkpoint(self, resume_from_checkpoint, model=None):
        config_file = os.path.join(resume_from_checkpoint, 'legamex_config.json')
        if not os.path.isfile(config_file):
            # Fall back to original behaviour for regular checkpoints
            super()._load_from_checkpoint(resume_from_checkpoint, model=model)
            return

        # Reconstruct model using our lightweight adapter files
        model = LegamexLoraModel.from_pretrained(resume_from_checkpoint)
        model.to(self.args.device)
        self.model = model
        self.model_wrapped = model

        # NOTE: optimizer/scheduler state is NOT loaded here. Trainer.train()
        # calls _load_from_checkpoint() BEFORE _inner_training_loop() creates
        # the optimizer and scheduler (self.optimizer / self.lr_scheduler are
        # still None at this point), which caused
        #     AttributeError: 'NoneType' object has no attribute 'load_state_dict'
        # Their state is restored in _load_optimizer_and_scheduler() below,
        # which Trainer calls right after create_optimizer_and_scheduler().

        # Reload trainer state (epoch, best metrics, etc.)
        trainer_state_path = os.path.join(resume_from_checkpoint, 'trainer_state.json')
        if os.path.isfile(trainer_state_path):
            self.state = TrainerState.load_from_json(trainer_state_path)

        # Sync LegameX training step
        if hasattr(self.model, 'set_training_step'):
            self.model.set_training_step(self.state.global_step)

        logger.info(f"Resumed LegameX model from {resume_from_checkpoint}")

    def _load_optimizer_and_scheduler(self, resume_from_checkpoint):
        """Restore optimizer/scheduler state for LegameX checkpoints.

        Trainer calls this after creating the optimizer and scheduler, so it
        is the correct place to load optimizer.pt / scheduler.pt.
        """
        config_file = os.path.join(resume_from_checkpoint, 'legamex_config.json')
        if not os.path.isfile(config_file):
            # Regular checkpoint: let the default implementation handle it
            super()._load_optimizer_and_scheduler(resume_from_checkpoint)
            return

        device = next(self.model.parameters()).device
        optimizer_path = os.path.join(resume_from_checkpoint, 'optimizer.pt')
        if self.optimizer is not None and os.path.isfile(optimizer_path):
            self.optimizer.load_state_dict(
                torch.load(optimizer_path, map_location=device)
            )
        scheduler_path = os.path.join(resume_from_checkpoint, 'scheduler.pt')
        if self.lr_scheduler is not None and os.path.isfile(scheduler_path):
            self.lr_scheduler.load_state_dict(
                torch.load(scheduler_path, map_location=device)
            )
        scaler_path = os.path.join(resume_from_checkpoint, 'scaler.pt')
        if getattr(self, 'scaler', None) is not None and os.path.isfile(scaler_path):
            self.scaler.load_state_dict(
                torch.load(scaler_path, map_location=device)
            )

        logger.info(f"Restored optimizer/scheduler state from {resume_from_checkpoint}")

class GateStepCallback(TrainerCallback):
    """Updates the LegameX training step at every optimisation step."""
    def on_step_end(self, args, state, control, **kwargs):
        model = kwargs['model']
        if hasattr(model, 'set_training_step'):
            model.set_training_step(state.global_step)
        return control
    
class WandbGateStatsCallback(TrainerCallback):
    """Logs gate weight statistics at every optimisation step."""
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        # Gather only the LegamexLora modules
        lora_modules = [m for m in model.legamex_modules.values()
                        if hasattr(m, 'gate')]  # or isinstance(m, LegamexLora)
        if not lora_modules:
            return

        # Gate scheduler alpha (same for all layers)
        first_module = lora_modules[0]
        alpha = first_module.gate_scheduler()

        # Aggregate gate weight statistics
        A_mean, A_norm = 0.0, 0.0
        B_mean, B_norm = 0.0, 0.0
        n = len(lora_modules)

        for m in lora_modules:
            A_mean += m.gate.A.weight.mean().item()
            A_norm += m.gate.A.weight.norm(p=2).item()
            B_mean += m.gate.B.weight.mean().item()
            B_norm += m.gate.B.weight.norm(p=2).item()

        wandb.log({
            'gate/scheduler_alpha': alpha,
            'gate/A_mean': A_mean / n,
            'gate/A_norm': A_norm / n,
            'gate/B_mean': B_mean / n,
            'gate/B_norm': B_norm / n,
            'step': state.global_step,
        })

In [16]:
# Calculate steps per epoch and max steps
steps_per_epoch = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS))
max_steps = steps_per_epoch * NUM_EPOCHS
print("Steps per epoch:", steps_per_epoch)
print("Max steps:", max_steps)

Steps per epoch: 941
Max steps: 9410


In [17]:
# Initialize wandb
wandb.init(
    project="legamex",
    name=run_name,
)

# Load the SQuAD metric (string-level Exact Match and F1)
squad_metric = evaluate.load('squad')

# Compute metrics for SQuAD (string-level Exact Match and F1 via HF evaluate)
def compute_metrics_squad(eval_pred):
    start_logits, end_logits = eval_pred.predictions

    # Defensive check: bookkeeping must be feature-for-feature aligned with
    # the eval predictions. A mismatch means the preprocessing cell that
    # builds _squad_eval_bookkeeping did not run (or ran on a different
    # val set) - re-run it before training.
    n_features = len(start_logits)
    n_bookkeeping = len(_squad_eval_bookkeeping['feature_offset_mappings'])
    if n_features != n_bookkeeping:
        raise RuntimeError(
            f'Eval produced {n_features} features but the SQuAD bookkeeping '
            f'only has {n_bookkeeping}. Re-run the dataset preprocessing '
            f'cell so the bookkeeping is rebuilt for the full validation set.'
        )

    # For each feature, find the best span and reconstruct predicted text
    feature_predictions = []
    for feat_idx in range(len(start_logits)):
        s_log = start_logits[feat_idx]
        e_log = end_logits[feat_idx]
        offsets = _squad_eval_bookkeeping['feature_offset_mappings'][feat_idx]
        context = _squad_eval_bookkeeping['feature_contexts'][feat_idx]
        ex_id = _squad_eval_bookkeeping['feature_example_ids'][feat_idx]

        # Identify context tokens (non-zero offset range)
        context_tokens = [
            tok for tok, (start, end) in enumerate(offsets)
            if start != 0 or end != 0
        ]

        if not context_tokens:
            feature_predictions.append({
                'id': ex_id,
                'prediction_text': '',
                'score': -float('inf'),
            })
            continue

        # Find best span using top-k candidates
        n_best = 20
        k = min(n_best, len(context_tokens))
        top_starts = sorted(context_tokens, key=lambda t: s_log[t], reverse=True)[:k]
        top_ends = sorted(context_tokens, key=lambda t: e_log[t], reverse=True)[:k]

        best_score = -float('inf')
        best_start = context_tokens[0]
        best_end = context_tokens[0]

        for start_tok in top_starts:
            for end_tok in top_ends:
                if end_tok < start_tok:
                    continue
                score = float(s_log[start_tok] + e_log[end_tok])
                if score > best_score:
                    best_score = score
                    best_start = start_tok
                    best_end = end_tok

        # Reconstruct text
        start_char = offsets[best_start][0]
        end_char = offsets[best_end][1]
        pred_text = context[start_char:end_char]

        feature_predictions.append({
            'id': ex_id,
            'prediction_text': pred_text,
            'score': best_score,
        })

    # Group by example_id, pick best prediction per example
    example_predictions = {}
    for pred in feature_predictions:
        ex_id = pred['id']
        if ex_id not in example_predictions or pred['score'] > example_predictions[ex_id]['score']:
            example_predictions[ex_id] = pred

    predictions = [
        {'id': ex_id, 'prediction_text': pred['prediction_text']}
        for ex_id, pred in example_predictions.items()
    ]

    references = _squad_eval_bookkeeping['references']

    if not predictions or not references:
        return {'f1': 0.0, 'exact_match': 0.0}

    metrics = squad_metric.compute(predictions=predictions, references=references) or {
        'f1': 0.0,
        'exact_match': 0.0,
    }

    return {
        'f1': float(metrics.get('f1', 0.0)),
        'exact_match': float(metrics.get('exact_match', 0.0)),
    }

# Set up the trainer
training_args = TrainingArguments(
    remove_unused_columns=False,
    
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='f1' if TASK == 'squad' else 'eval_loss',
    greater_is_better=True if TASK == 'squad' else False,

    # Restore callback states (e.g. EarlyStoppingCallback's
    # early_stopping_patience_counter) from the checkpoint on resume.
    # Without this, the counter silently resets to 0 on every resume
    # (transformers 4.53.3 default), making early stopping more lenient.
    restore_callback_states_from_checkpoint=True,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
    
    # TEMP: Do not save and upload checkpoints for testing
    # save_strategy='no', 
    # load_best_model_at_end=False,
)
trainer = LegamexTrainer(
    model=legamex_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics_squad if TASK == 'squad' else None,
    # label_names=['labels'],
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            # early_stopping_threshold = 0.001,
        ),
        GateStepCallback(),
        WandbGateStatsCallback(),
    ],
)
trainer.label_names = label_names

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [18]:
# Start training
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 1,494,530 / 280,274,690


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss,F1,Exact Match
6300,1.003200,1.253322,78.613726,70.400000


Could not locate the best model at XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047/checkpoint-4800/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.
